# 収量予測モデルの再構築と地域ごとの閾値

load_regressionDF_simple と同じデータ・特徴量・XGBoostモデルを再構築し、**各地域（国×穀物）ごとの収量閾値**（平均・中央値・パーセンタイル）を算出します。

In [ ]:
import pandas as pd
import numpy as np
import os

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
file_path = os.path.join(base_dir, 'regression_df_20260112_233447.xlsx')

print("データを読み込んでいます...")
regressionDF = pd.read_excel(file_path)
print("✓ データの読み込みが完了しました。")
print(f"データ形状: {regressionDF.shape}")

データを読み込んでいます...
✓ データの読み込みが完了しました。
データ形状: (16388, 8)


In [ ]:
# 特徴量の準備（load_regressionDF_simple と同じ）
from sklearn.model_selection import train_test_split

climate_features = ['SPEI', 'GSL', 'Hurs', 'TXX']
area_dummies = pd.get_dummies(regressionDF['Area'], prefix='Area')
item_dummies = pd.get_dummies(regressionDF['Item'], prefix='Item')
year_feature = regressionDF[['Year']].copy()

X = pd.concat([
    regressionDF[climate_features],
    area_dummies,
    item_dummies,
    year_feature
], axis=1)
y = regressionDF['Yield'].copy()

print(f"特徴量: {X.shape}, 目的変数: {y.shape}")

特徴量: (16388, 164), 目的変数: (16388,)


In [ ]:
# XGBoostモデルを再構築（load_regressionDF_simple と同じ best_params で全データ学習）
import xgboost as xgb

best_params = {
    'colsample_bytree': 1.0,
    'learning_rate': 0.2,
    'max_depth': 5,
    'n_estimators': 300,
    'subsample': 0.9,
    'random_state': 42,
    'n_jobs': -1
}

model = xgb.XGBRegressor(**best_params)
model.fit(X, y)
print("✓ モデル学習完了（全データで学習）")

✓ モデル学習完了（全データで学習）


## 北米・中南米・オセアニア・南ヨーロッパ・東南アジアの5地域で集計

国（Area）を **北米 / 中南米 / オセアニア / 南ヨーロッパ / 東南アジア** に分類し、地域ごとの閾値とサマリーを算出します。  
※ 本節は**5地域に含まれる国のみ**が対象です。**65ヶ国版**（optimal_crop 対象国のみ）は下記「65ヶ国版」セクションで行います。

In [ ]:
# 国（Area）→ 北米・中南米・オセアニア・南ヨーロッパ・東南アジア のマッピング（FAO国名に合わせて定義）
REGION_MAP = {
    # 北米
    'United States of America': '北米',
    'Canada': '北米',
    'Mexico': '北米',
    # 中南米（中央アメリカ・カリブ・南米）
    'Belize': '中南米', 'Guatemala': '中南米', 'Honduras': '中南米', 'El Salvador': '中南米',
    'Nicaragua': '中南米', 'Costa Rica': '中南米', 'Panama': '中南米',
    'Cuba': '中南米', 'Jamaica': '中南米', 'Haiti': '中南米', 'Dominican Republic': '中南米',
    'Trinidad and Tobago': '中南米', 'Bahamas': '中南米', 'Barbados': '中南米',
    'Saint Lucia': '中南米', 'Grenada': '中南米', 'Antigua and Barbuda': '中南米',
    'Saint Vincent and the Grenadines': '中南米', 'Saint Kitts and Nevis': '中南米',
    'Dominica': '中南米', 'Suriname': '中南米', 'Guyana': '中南米',
    'Colombia': '中南米', 'Venezuela (Bolivarian Republic of)': '中南米',
    'Ecuador': '中南米', 'Peru': '中南米', 'Brazil': '中南米',
    'Bolivia (Plurinational State of)': '中南米', 'Chile': '中南米',
    'Argentina': '中南米', 'Uruguay': '中南米', 'Paraguay': '中南米',
    # オセアニア
    'Australia': 'オセアニア', 'New Zealand': 'オセアニア',
    # 南ヨーロッパ（地中海・イベリア・イタリア・ギリシャ・バルカン）
    'Albania': '南ヨーロッパ', 'Bosnia and Herzegovina': '南ヨーロッパ', 'Croatia': '南ヨーロッパ',
    'Cyprus': '南ヨーロッパ', 'Greece': '南ヨーロッパ', 'Italy': '南ヨーロッパ', 'Malta': '南ヨーロッパ',
    'Montenegro': '南ヨーロッパ', 'North Macedonia': '南ヨーロッパ', 'Portugal': '南ヨーロッパ',
    'Serbia': '南ヨーロッパ', 'Slovenia': '南ヨーロッパ', 'Spain': '南ヨーロッパ',
    # 東南アジア
    'Bangladesh': '東南アジア', 'Brunei Darussalam': '東南アジア', 'Brunei': '東南アジア', 'Cambodia': '東南アジア',
    'Indonesia': '東南アジア', "Lao People's Democratic Republic": '東南アジア', 'Malaysia': '東南アジア',
    'Myanmar': '東南アジア', 'Philippines': '東南アジア', 'Singapore': '東南アジア',
    'Thailand': '東南アジア', 'Timor-Leste': '東南アジア', 'Viet Nam': '東南アジア', 'Vietnam': '東南アジア',
}
REGIONS_LIST = ['北米', '中南米', 'オセアニア', '南ヨーロッパ', '東南アジア']

# 元データに地域を付与（5地域に含まれない国は NaN にし、後でフィルタ可能に）
regressionDF['Region'] = regressionDF['Area'].map(REGION_MAP)

# 5地域に含まれるデータのみ
df_3regions = regressionDF[regressionDF['Region'].notna()].copy()
print("北米・中南米・オセアニア・南ヨーロッパ・東南アジアの5地域:")
print(df_3regions.groupby('Region')['Area'].nunique())
print(f"\n5地域の総行数: {len(df_3regions)} / 全体: {len(regressionDF)}")

北米・中南米・オセアニア・南ヨーロッパ・東南アジアの5地域:
Region
オセアニア      2
中南米       27
北米         3
南ヨーロッパ    13
東南アジア     10
Name: Area, dtype: int64

5地域の総行数: 5988 / 全体: 16388


In [ ]:
# 5地域ごとの国数・行数
print("5地域ごとの国数・行数:")
print(df_3regions.groupby('Region').agg(国数=('Area', 'nunique'), 行数=('Yield', 'count')))
print("\n5地域に含まれる国（Area）:")
for r in REGIONS_LIST:
    countries = df_3regions[df_3regions['Region'] == r]['Area'].unique().tolist()
    if countries:
        print(f"  {r}: {sorted(countries)}")

5地域ごとの国数・行数:
        国数    行数
Region          
オセアニア    2   262
中南米     27  3097
北米       3   421
南ヨーロッパ  13  1107
東南アジア   10  1101

5地域に含まれる国（Area）:
  北米: ['Canada', 'Mexico', 'United States of America']
  中南米: ['Antigua and Barbuda', 'Argentina', 'Barbados', 'Belize', 'Brazil', 'Chile', 'Colombia', 'Costa Rica', 'Cuba', 'Dominica', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Grenada', 'Guatemala', 'Guyana', 'Haiti', 'Honduras', 'Jamaica', 'Nicaragua', 'Panama', 'Paraguay', 'Peru', 'Saint Vincent and the Grenadines', 'Suriname', 'Trinidad and Tobago', 'Uruguay']
  オセアニア: ['Australia', 'New Zealand']
  南ヨーロッパ: ['Albania', 'Bosnia and Herzegovina', 'Croatia', 'Cyprus', 'Greece', 'Italy', 'Malta', 'Montenegro', 'North Macedonia', 'Portugal', 'Serbia', 'Slovenia', 'Spain']
  東南アジア: ['Bangladesh', 'Brunei Darussalam', 'Cambodia', 'Indonesia', "Lao People's Democratic Republic", 'Malaysia', 'Myanmar', 'Philippines', 'Thailand', 'Timor-Leste']


In [ ]:
# 5地域ごとの収量閾値
g_region = df_3regions.groupby('Region')['Yield']
thresholds_3regions = pd.DataFrame({
    '閾値_平均': g_region.mean(),
    '閾値_中央値': g_region.median(),
    '閾値_25%点': g_region.quantile(0.25),
    '閾値_75%点': g_region.quantile(0.75),
    '観測数': g_region.count()
}).reset_index().round(2)
thresholds_3regions

,Region,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,5310.34,5297.85,2505.1,7650.80,262
1,中南米,2464.90,1998.30,1314.9,3169.90,3097
2,北米,4187.96,3913.80,2316.7,5477.80,421
3,南ヨーロッパ,4192.08,3800.90,2430.8,5689.35,1107
4,東南アジア,2051.57,1797.00,1200.0,2698.60,1101


In [ ]:
# 5地域 × 穀物（Item）ごとの収量閾値
g_region_item = df_3regions.groupby(['Region', 'Item'])['Yield']
thresholds_region_item = pd.DataFrame({
    '閾値_平均': g_region_item.mean(),
    '閾値_中央値': g_region_item.median(),
    '閾値_25%点': g_region_item.quantile(0.25),
    '閾値_75%点': g_region_item.quantile(0.75),
    '観測数': g_region_item.count()
}).reset_index().round(2)
thresholds_region_item

,Region,Item,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,Maize (corn),6317.15,5928.30,3579.80,9119.60,105
1,オセアニア,Rice,7427.86,7305.95,6293.38,8363.10,52
2,オセアニア,Wheat,3254.86,2648.20,1451.50,4243.10,105
3,中南米,Maize (corn),2146.03,1591.75,1193.65,2452.68,1414
4,中南米,Rice,3264.99,3120.90,2208.55,4086.55,1159
5,中南米,Wheat,1555.67,1347.05,924.70,1970.45,524
6,北米,Maize (corn),5115.19,5310.95,2498.43,7119.15,158
7,北米,Rice,4881.01,4839.90,3741.80,6176.30,105
8,北米,Wheat,2800.16,2530.00,2042.62,3562.80,158
9,南ヨーロッパ,Maize (corn),5360.74,4767.95,3371.82,7328.22,368


In [ ]:
# 5地域の閾値をCSVで保存
path_5regions = os.path.join(base_dir, 'thresholds_5regions.csv')
path_5regions_item = os.path.join(base_dir, 'thresholds_5regions_by_item.csv')
thresholds_3regions.to_csv(path_5regions, index=False, encoding='utf-8-sig')
thresholds_region_item.to_csv(path_5regions_item, index=False, encoding='utf-8-sig')
print(f"✓ 5地域ごと: {path_5regions}")
print(f"✓ 5地域×穀物ごと: {path_5regions_item}")

✓ 5地域ごと: C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\thresholds_5regions.csv
✓ 5地域×穀物ごと: C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\thresholds_5regions_by_item.csv


## 各変数（SPEI, GSL, Hurs, TXX）の閾値

気候変数ごとに、5地域（北米・中南米・オセアニア・南ヨーロッパ・東南アジア）での **閾値_平均・閾値_中央値・閾値_25%点・閾値_75%点** を算出します。予測や異常判定の基準値として利用できます。

In [ ]:
# 各変数（SPEI, GSL, Hurs, TXX）の5地域ごと閾値
climate_features = ['SPEI', 'GSL', 'Hurs', 'TXX']
rows = []
for var in climate_features:
    g = df_3regions.groupby('Region')[var]
    t = pd.DataFrame({
        '閾値_平均': g.mean(),
        '閾値_中央値': g.median(),
        '閾値_25%点': g.quantile(0.25),
        '閾値_75%点': g.quantile(0.75),
        '観測数': g.count()
    }).reset_index()
    t.insert(1, '変数', var)
    rows.append(t)
thresholds_by_variable = pd.concat(rows, ignore_index=True).round(4)
thresholds_by_variable

,Region,変数,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,SPEI,0.0967,0.09,-0.2200,0.4175,262
1,中南米,SPEI,0.1848,0.18,-0.2800,0.6100,3097
2,北米,SPEI,-0.3247,-0.31,-0.5600,-0.1200,421
3,南ヨーロッパ,SPEI,0.2878,0.31,-0.2000,0.7900,1107
4,東南アジア,SPEI,-0.0206,-0.03,-0.3800,0.3800,1101
5,オセアニア,GSL,340.7134,359.45,313.5300,359.5700,262
6,中南米,GSL,352.3806,360.00,360.0000,360.0000,3097
7,北米,GSL,260.0070,249.59,135.7400,359.6300,421
8,南ヨーロッパ,GSL,316.2346,322.54,294.5450,339.4900,1107
9,東南アジア,GSL,359.7241,360.00,360.0000,360.0000,1101


In [41]:
# 地域×穀物（Item）別の各変数閾値
rows_ri = []
for var in climate_features:
    g = df_3regions.groupby(['Region', 'Item'])[var]
    t = pd.DataFrame({
        '閾値_平均': g.mean(),
        '閾値_中央値': g.median(),
        '閾値_25%点': g.quantile(0.25),
        '閾値_75%点': g.quantile(0.75),
        '観測数': g.count()
    }).reset_index()
    t.insert(2, '変数', var)
    rows_ri.append(t)
thresholds_by_variable_item = pd.concat(rows_ri, ignore_index=True).round(4)
thresholds_by_variable_item

,Region,Item,変数,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,Maize (corn),SPEI,0.0775,0.090,-0.2400,0.3700,105
1,オセアニア,Rice,SPEI,0.1742,0.100,-0.1450,0.5425,52
2,オセアニア,Wheat,SPEI,0.0775,0.090,-0.2400,0.3700,105
3,中南米,Maize (corn),SPEI,0.2532,0.250,-0.2075,0.6900,1414
4,中南米,Rice,SPEI,0.1805,0.170,-0.2800,0.6000,1159
5,中南米,Wheat,SPEI,0.0100,0.030,-0.3900,0.4225,524
6,北米,Maize (corn),SPEI,-0.3446,-0.330,-0.5775,-0.1500,158
7,北米,Rice,SPEI,-0.2650,-0.250,-0.4700,-0.0800,105
8,北米,Wheat,SPEI,-0.3446,-0.330,-0.5775,-0.1500,158
9,南ヨーロッパ,Maize (corn),SPEI,0.2765,0.305,-0.2100,0.8000,368


In [ ]:
# 各変数閾値をCSVで保存
path_var = os.path.join(base_dir, 'thresholds_by_variable_5regions.csv')
path_var_item = os.path.join(base_dir, 'thresholds_by_variable_5regions_by_item.csv')
thresholds_by_variable.to_csv(path_var, index=False, encoding='utf-8-sig')
thresholds_by_variable_item.to_csv(path_var_item, index=False, encoding='utf-8-sig')
print(f"✓ 地域×変数: {path_var}")
print(f"✓ 地域×穀物×変数: {path_var_item}")

## 2100年データとの照合：どの変数が影響が多いか

将来シナリオ（SSP245・SSP370、2096–2099年平均を2100年として利用）の気候値を、上で求めた**地域別の各変数閾値**と照らし合わせ、歴史的な範囲からの乖離度を算出します。乖離が大きい変数ほど、収量への影響が強く現れやすいと解釈できます。

In [ ]:
# 2100年気候データの読み込み（SSP245 / SSP370）
f245 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP245_nfuture_2096-2099_mean_20260129_043901.csv'))
f370 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP370_bfuture_2096-2099_mean_20260129_043854.csv'))

# 5地域に含まれる国のみに限定し、地域ラベルを付与
areas_3regions = df_3regions[['Area', 'Region']].drop_duplicates()
f245_r = f245[f245['Area'].isin(areas_3regions['Area'])].merge(areas_3regions, on='Area')
f370_r = f370[f370['Area'].isin(areas_3regions['Area'])].merge(areas_3regions, on='Area')

print("2100年データ（5地域の国のみ）:")
print(f"  SSP245: {len(f245_r)} ヶ国")
print(f"  SSP370: {len(f370_r)} ヶ国")
print(f"  変数: {climate_features}")

2100年データ（5地域の国のみ）:
  SSP245: 55 ヶ国
  SSP370: 55 ヶ国
  変数: ['SPEI', 'GSL', 'Hurs', 'TXX']


In [ ]:
# 地域ごとの2100年平均値と、歴史的閾値との差・乖離度を算出
# thresholds_by_variable: Region, 変数, 閾値_平均, 閾値_中央値, 閾値_25%点, 閾値_75%点, 観測数
def compare_future_to_thresholds(future_df, scenario_name, thresh_df):
    thresh = thresh_df.set_index(['Region', '変数'])
    results = []
    for region in REGIONS_LIST:
        f_region = future_df[future_df['Region'] == region]
        if len(f_region) == 0:
            continue
        for var in climate_features:
            mean_2100 = f_region[var].mean()
            row = thresh.loc[(region, var)]
            mean_hist = row['閾値_平均']
            p25, p75 = row['閾値_25%点'], row['閾値_75%点']
            iqr = (p75 - p25) if (p75 != p25) else 1.0  # 範囲で正規化
            deviation = mean_2100 - mean_hist
            deviation_iqr = deviation / iqr  # IQR何個分の乖離か
            outside_range = (mean_2100 < p25) or (mean_2100 > p75)
            results.append({
                'Scenario': scenario_name,
                'Region': region,
                '変数': var,
                '2100平均': mean_2100,
                '歴史的平均': mean_hist,
                '差_2100minus歴史': deviation,
                '差_IQR単位': deviation_iqr,
                '歴史的25%点': p25,
                '歴史的75%点': p75,
                '範囲外': outside_range
            })
    return pd.DataFrame(results)

compare_245 = compare_future_to_thresholds(f245_r, 'SSP245', thresholds_by_variable)
compare_370 = compare_future_to_thresholds(f370_r, 'SSP370', thresholds_by_variable)
comparison_2100 = pd.concat([compare_245, compare_370], ignore_index=True).round(4)
comparison_2100

,Scenario,Region,変数,2100平均,歴史的平均,差_2100minus歴史,差_IQR単位,歴史的25%点,歴史的75%点,範囲外
0,SSP245,北米,SPEI,0.1808,-0.3247,0.5055,1.1489,-0.5600,-0.1200,True
1,SSP245,北米,GSL,265.7283,260.0070,5.7213,0.0256,135.7400,359.6300,False
2,SSP245,北米,Hurs,62.6283,62.8514,-0.2231,-0.0151,57.6100,72.4200,False
3,SSP245,北米,TXX,36.2717,31.8674,4.4043,0.5391,26.9500,35.1200,True
4,SSP245,中南米,SPEI,-0.0578,0.1848,-0.2426,-0.2726,-0.2800,0.6100,False
5,SSP245,中南米,GSL,356.6131,352.3806,4.2325,4.2325,360.0000,360.0000,True
6,SSP245,中南米,Hurs,76.0437,75.5765,0.4672,0.0620,73.5600,81.0900,False
7,SSP245,中南米,TXX,34.7027,31.6081,3.0946,0.6381,29.1500,34.0000,True
8,SSP245,オセアニア,SPEI,0.0125,0.0967,-0.0842,-0.1321,-0.2200,0.4175,False
9,SSP245,オセアニア,GSL,346.9125,340.7134,6.1991,0.1346,313.5300,359.5700,False


In [ ]:
# 2100年 vs 歴史的閾値の表（シナリオ・地域・変数別）を表示
cols = ['Scenario', 'Region', '変数', '2100平均', '歴史的平均', '差_2100minus歴史', '差_IQR単位',
        '歴史的25%点', '歴史的75%点', '範囲外']
display_df = comparison_2100[cols].copy()
display_df.index = range(len(display_df))
print("2100年 vs 歴史的閾値（シナリオ・地域・変数別）")
display(display_df)

2100年 vs 歴史的閾値（シナリオ・地域・変数別）


,Scenario,Region,変数,2100平均,歴史的平均,差_2100minus歴史,差_IQR単位,歴史的25%点,歴史的75%点,範囲外
0,SSP245,北米,SPEI,0.1808,-0.3247,0.5055,1.1489,-0.5600,-0.1200,True
1,SSP245,北米,GSL,265.7283,260.0070,5.7213,0.0256,135.7400,359.6300,False
2,SSP245,北米,Hurs,62.6283,62.8514,-0.2231,-0.0151,57.6100,72.4200,False
3,SSP245,北米,TXX,36.2717,31.8674,4.4043,0.5391,26.9500,35.1200,True
4,SSP245,中南米,SPEI,-0.0578,0.1848,-0.2426,-0.2726,-0.2800,0.6100,False
5,SSP245,中南米,GSL,356.6131,352.3806,4.2325,4.2325,360.0000,360.0000,True
6,SSP245,中南米,Hurs,76.0437,75.5765,0.4672,0.0620,73.5600,81.0900,False
7,SSP245,中南米,TXX,34.7027,31.6081,3.0946,0.6381,29.1500,34.0000,True
8,SSP245,オセアニア,SPEI,0.0125,0.0967,-0.0842,-0.1321,-0.2200,0.4175,False
9,SSP245,オセアニア,GSL,346.9125,340.7134,6.1991,0.1346,313.5300,359.5700,False


: 

In [30]:
# 変数ごとの「影響の大きさ」を要約：差の絶対値（IQR単位）の平均でランク付け
impact = comparison_2100.groupby('変数').agg(
    差_IQR絶対値の平均=('差_IQR単位', lambda x: np.abs(x).mean()),
    範囲外になる回数=('範囲外', 'sum'),
    差_絶対値の平均=('差_2100minus歴史', lambda x: np.abs(x).mean())
).round(4)
impact['影響度順位_IQR'] = impact['差_IQR絶対値の平均'].rank(ascending=False).astype(int)
impact = impact.sort_values('差_IQR絶対値の平均', ascending=False)
print("=== 変数ごとの影響の大きさ（2100年 vs 歴史的閾値）===")
print("差_IQR絶対値の平均: 歴史的なばらつき（25–75%範囲）の何倍分、2100年がずれているか。大きいほど影響が大きい。")
print(impact)

=== 変数ごとの影響の大きさ（2100年 vs 歴史的閾値）===
差_IQR絶対値の平均: 歴史的なばらつき（25–75%範囲）の何倍分、2100年がずれているか。大きいほど影響が大きい。
      差_IQR絶対値の平均  範囲外になる回数  差_絶対値の平均  影響度順位_IQR
変数                                              
GSL        1.6153         2    7.2515          1
SPEI       0.9259         4    0.5376          2
TXX        0.5104         4    3.7736          3
Hurs       0.0542         0    1.1020          4


In [18]:
# 地域・シナリオ別の乖離（差_IQR単位）をピボット表示
pivot_iqr = comparison_2100.pivot_table(
    index=['Region', '変数'], columns='Scenario', values='差_IQR単位', aggfunc='mean'
).round(3)
print("差_IQR単位（正: 2100が歴史より高い、負: 低い）:")
print(pivot_iqr)

差_IQR単位（正: 2100が歴史より高い、負: 低い）:
Scenario     SSP245  SSP370
Region 変数                  
オセアニア  GSL    8.570  11.065
       Hurs   0.113   0.110
       SPEI  -0.312  -0.269
       TXX    0.146   0.264
中南米    GSL    4.232   5.025
       Hurs   0.062  -0.019
       SPEI  -0.273  -1.115
       TXX    0.638   0.869
北米     GSL    0.026   0.055
       Hurs  -0.015  -0.087
       SPEI   1.149   2.225
       TXX    0.539   0.756


In [ ]:
# 2100年 vs 閾値 の比較結果を保存
comparison_2100.to_csv(os.path.join(base_dir, 'comparison_2100_vs_thresholds.csv'), index=False, encoding='utf-8-sig')
impact.to_csv(os.path.join(base_dir, 'variable_impact_rank_2100.csv'), encoding='utf-8-sig')
print("✓ comparison_2100_vs_thresholds.csv")
print("✓ variable_impact_rank_2100.csv")

### 考察（どの変数が影響が多いか）

- **差_IQR絶対値の平均**が大きい変数ほど、2100年の値が歴史的な「普通の範囲」（25–75%点の幅）から大きく外れており、**収量への影響が強く出やすい**と解釈できる。
- **TXX（最高気温）**: 温暖化により多くの地域で上昇し、歴史的範囲を上回りやすい。差_IQRが大きい場合は熱ストレスによる減収リスクが高い。
- **SPEI（干ばつ指標）**: 負の方向への乖離が大きい地域では干ばつリスクの増加、正なら湿潤化の可能性。地域差が大きい。
- **GSL（生育可能日数）**: 増加する場合が多く、寒冷地では収量増の要因にもなるが、既に長い地域では効果が限定的。
- **Hurs（相対湿度）**: 気温上昇に伴い多くの地域で変化し、蒸散・病害リスクと関連する。

上記の「変数ごとの影響度順位」と「地域・シナリオ別の乖離」表を合わせて、**5地域（北米・中南米・オセアニア・南ヨーロッパ・東南アジア）のどこで、どの変数を特に注視すべきか**（例: 中南米のTXX上昇、オセアニアのSPEI低下など）を判断できます。SSP370はSSP245より変化が大きいため、影響度も一般に高くなります。

## 65ヶ国版（対象国のみに絞った分析）

optimal_crop_65countries_5scenarios_rice088.csv に含まれる**65ヶ国のみ**を対象に、上記と同様の閾値・各変数閾値・2100年比較を行います。5地域の集計は「65ヶ国のうち北米・中南米・オセアニア・南ヨーロッパ・東南アジアに属する国」で再計算します。

In [ ]:
# 65ヶ国に入らなかった国のリスト（回帰データにある全国のうち65に含まれない国）
opt65 = pd.read_csv(os.path.join(base_dir, 'optimal_crop_65countries_5scenarios_rice088.csv'))
areas_65 = sorted(opt65['Area'].unique().tolist())
areas_all = sorted(regressionDF['Area'].unique().tolist())
areas_not_in_65 = sorted(set(areas_all) - set(areas_65))
print(f"回帰データの国数: {len(areas_all)}")
print(f"65ヶ国に含まれる: {len(areas_65)}")
print(f"65ヶ国に入らなかった国: {len(areas_not_in_65)} ヶ国\n")
print("【65ヶ国に入らなかった国のリスト】")
for i, a in enumerate(areas_not_in_65, 1):
    print(f"  {i:2d}. {a}")

回帰データの国数: 156
65ヶ国に含まれる: 65
65ヶ国に入らなかった国: 91 ヶ国

【65ヶ国に入らなかった国のリスト】
   1. Afghanistan
   2. Albania
   3. Algeria
   4. Angola
   5. Antigua and Barbuda
   6. Azerbaijan
   7. Barbados
   8. Belize
   9. Benin
  10. Bhutan
  11. Bosnia and Herzegovina
  12. Botswana
  13. Brazil
  14. Brunei Darussalam
  15. Bulgaria
  16. Burkina Faso
  17. Cabo Verde
  18. Cambodia
  19. Cameroon
  20. Central African Republic
  21. Chad
  22. Congo
  23. Croatia
  24. Cuba
  25. Cyprus
  26. Côte d'Ivoire
  27. Denmark
  28. Djibouti
  29. Dominica
  30. Eritrea
  31. Estonia
  32. Ethiopia
  33. Finland
  34. Gabon
  35. Georgia
  36. Grenada
  37. Guinea
  38. Hungary
  39. India
  40. Indonesia
  41. Iraq
  42. Ireland
  43. Jamaica
  44. Jordan
  45. Kenya
  46. Lao People's Democratic Republic
  47. Latvia
  48. Lebanon
  49. Lesotho
  50. Liberia
  51. Lithuania
  52. Luxembourg
  53. Madagascar
  54. Malawi
  55. Malaysia
  56. Maldives
  57. Mali
  58. Malta
  59. Mauritius
  60. Mongolia
  

In [ ]:
# 65ヶ国に入らなかった国を色塗りした地図（含まれない=オレンジ、含まれる=青）
import plotly.express as px

# 地図用：全国に 65に含まれる=1 / 含まれない=0 を付与
map_df = pd.DataFrame({'Area': areas_all})
map_df['65に含まれる'] = map_df['Area'].isin(areas_65).astype(int)
map_df['ラベル'] = map_df['65に含まれる'].map({1: '65ヶ国に含まれる', 0: '65ヶ国に入らなかった'})

# Plotly用の国名変換（地図で認識されやすい名前に）
name_map = {
    'United States of America': 'United States',
    'Republic of Korea': 'South Korea',
    "Democratic People's Republic of Korea": 'North Korea',
    'Russian Federation': 'Russia',
    'Iran (Islamic Republic of)': 'Iran',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'United Republic of Tanzania': 'Tanzania',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Syrian Arab Republic': 'Syria',
    'Czechia': 'Czech Republic',
    'The former Yugoslav Republic of Macedonia': 'North Macedonia',
    "Lao People's Democratic Republic": 'Laos',
    'Brunei Darussalam': 'Brunei',
    'Democratic Republic of the Congo': 'Democratic Republic of the Congo',
    'Republic of Moldova': 'Moldova',
    'Viet Nam': 'Vietnam',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    "Côte d'Ivoire": 'Ivory Coast',
}
map_df['Country'] = map_df['Area'].replace(name_map)

fig = px.choropleth(
    map_df,
    locations='Country',
    locationmode='country names',
    color='ラベル',
    color_discrete_map={'65ヶ国に含まれる': 'steelblue', '65ヶ国に入らなかった': 'coral'},
    hover_name='Area',
    hover_data={'65に含まれる': True, 'Country': False},
    title='65ヶ国対象：入らなかった国（オレンジ）・含まれる国（青）'
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=500)
fig.show()

C:\Users\xyz19\AppData\Local\Temp\ipykernel_17876\1241774470.py:32: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [ ]:
# 65ヶ国リストの読み込み（optimal_crop_65countries の Area のユニーク）
opt65 = pd.read_csv(os.path.join(base_dir, 'optimal_crop_65countries_5scenarios_rice088.csv'))
areas_65 = sorted(opt65['Area'].unique().tolist())
print(f"65ヶ国リスト: {len(areas_65)} ヶ国")
# 回帰データを65ヶ国に絞り、地域ラベルを付与
df_65 = regressionDF[regressionDF['Area'].isin(areas_65)].copy()
df_65['Region'] = df_65['Area'].map(REGION_MAP)
# 65ヶ国のうち5地域に属する国
df_65_3regions = df_65[df_65['Region'].notna()].copy()
print(f"65ヶ国の行数: {len(df_65)}")
print(f"65ヶ国のうち5地域（北米・中南米・オセアニア・南ヨーロッパ・東南アジア）に属する国: {df_65_3regions['Area'].nunique()} ヶ国")
print(df_65_3regions.groupby('Region')['Area'].nunique())

65ヶ国リスト: 65 ヶ国
65ヶ国の行数: 7958
65ヶ国のうち5地域（北米・中南米・オセアニア・南ヨーロッパ・東南アジア）に属する国: 29 ヶ国
Region
オセアニア      2
中南米       17
北米         3
南ヨーロッパ     5
東南アジア      2
Name: Area, dtype: int64


In [34]:
# 65ヶ国版：5地域（65ヶ国中）の収量閾値
g65 = df_65_3regions.groupby('Region')['Yield']
thresholds_3regions_65 = pd.DataFrame({
    '閾値_平均': g65.mean(),
    '閾値_中央値': g65.median(),
    '閾値_25%点': g65.quantile(0.25),
    '閾値_75%点': g65.quantile(0.75),
    '観測数': g65.count()
}).reset_index().round(2)
# 65ヶ国版：5地域×穀物
g65_item = df_65_3regions.groupby(['Region', 'Item'])['Yield']
thresholds_region_item_65 = pd.DataFrame({
    '閾値_平均': g65_item.mean(),
    '閾値_中央値': g65_item.median(),
    '閾値_25%点': g65_item.quantile(0.25),
    '閾値_75%点': g65_item.quantile(0.75),
    '観測数': g65_item.count()
}).reset_index().round(2)
print("65ヶ国版・5地域ごとの収量閾値:")
thresholds_3regions_65

65ヶ国版・5地域ごとの収量閾値:


,Region,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,5310.34,5297.85,2505.10,7650.80,262
1,中南米,2552.07,2075.50,1334.12,3418.42,2206
2,北米,4187.96,3913.80,2316.70,5477.80,421
3,南ヨーロッパ,3807.72,3619.75,2658.62,4680.32,326
4,東南アジア,1976.07,1825.40,1095.90,2873.90,285


In [37]:
# 65ヶ国版：各変数（SPEI, GSL, Hurs, TXX）の地域別閾値
rows65 = []
for var in climate_features:
    g = df_65_3regions.groupby('Region')[var]
    t = pd.DataFrame({
        '閾値_平均': g.mean(),
        '閾値_中央値': g.median(),
        '閾値_25%点': g.quantile(0.25),
        '閾値_75%点': g.quantile(0.75),
        '観測数': g.count()
    }).reset_index()
    t.insert(1, '変数', var)
    rows65.append(t)
thresholds_by_variable_65 = pd.concat(rows65, ignore_index=True).round(4)
print("65ヶ国版・地域×変数ごとの閾値:")
thresholds_by_variable_65

65ヶ国版・地域×変数ごとの閾値:


,Region,変数,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,オセアニア,SPEI,0.0967,0.09,-0.2200,0.4175,262
1,中南米,SPEI,0.1372,0.12,-0.3400,0.5700,2206
2,北米,SPEI,-0.3247,-0.31,-0.5600,-0.1200,421
3,オセアニア,GSL,340.7134,359.45,313.5300,359.5700,262
4,中南米,GSL,349.3046,360.00,360.0000,360.0000,2206
5,北米,GSL,260.0070,249.59,135.7400,359.6300,421
6,オセアニア,Hurs,59.7522,47.59,46.8200,78.5100,262
7,中南米,Hurs,75.4357,76.85,73.3400,81.2100,2206
8,北米,Hurs,62.8514,61.48,57.6100,72.4200,421
9,オセアニア,TXX,33.5107,40.48,22.7525,41.0500,262


In [6]:
# 65ヶ国版：2100年 vs 歴史的閾値の表を表示
cols = ['Scenario', 'Region', '変数', '2100平均', '歴史的平均', '差_2100minus歴史', '差_IQR単位',
        '歴史的25%点', '歴史的75%点', '範囲外']
display_df_65 = comparison_2100_65[cols].copy()
display_df_65.index = range(len(display_df_65))
print("65ヶ国版：2100年 vs 歴史的閾値（シナリオ・地域・変数別）")
display(display_df_65)

NameError: name 'comparison_2100_65' is not defined

In [40]:
# 65ヶ国版：2100年データを65ヶ国に絞って閾値と比較
areas_65_3regions = df_65_3regions[['Area', 'Region']].drop_duplicates()
f245_65 = f245[f245['Area'].isin(areas_65_3regions['Area'])].merge(areas_65_3regions, on='Area')
f370_65 = f370[f370['Area'].isin(areas_65_3regions['Area'])].merge(areas_65_3regions, on='Area')
compare_245_65 = compare_future_to_thresholds(f245_65, 'SSP245', thresholds_by_variable_65)
compare_370_65 = compare_future_to_thresholds(f370_65, 'SSP370', thresholds_by_variable_65)
comparison_2100_65 = pd.concat([compare_245_65, compare_370_65], ignore_index=True).round(4)
# 変数ごとの影響度（65ヶ国版）
impact_65 = comparison_2100_65.groupby('変数').agg(
    差_IQR絶対値の平均=('差_IQR単位', lambda x: np.abs(x).mean()),
    範囲外になる回数=('範囲外', 'sum')
).round(4)
impact_65['影響度順位'] = impact_65['差_IQR絶対値の平均'].rank(ascending=False).astype(int)
impact_65 = impact_65.sort_values('差_IQR絶対値の平均', ascending=False)
print("65ヶ国版・2100年 vs 閾値（変数ごと影響度）:")
impact_65

65ヶ国版・2100年 vs 閾値（変数ごと影響度）:


,差_IQR絶対値の平均,範囲外になる回数,影響度順位
変数,,,
GSL,2.0546,2,1
SPEI,0.8476,4,2
TXX,0.5891,4,3
Hurs,0.0765,0,4


In [ ]:
# 65ヶ国版の結果をCSVで保存
thresholds_3regions_65.to_csv(os.path.join(base_dir, 'thresholds_5regions_65countries.csv'), index=False, encoding='utf-8-sig')
thresholds_region_item_65.to_csv(os.path.join(base_dir, 'thresholds_5regions_by_item_65countries.csv'), index=False, encoding='utf-8-sig')
thresholds_by_variable_65.to_csv(os.path.join(base_dir, 'thresholds_by_variable_5regions_65countries.csv'), index=False, encoding='utf-8-sig')
comparison_2100_65.to_csv(os.path.join(base_dir, 'comparison_2100_vs_thresholds_65countries.csv'), index=False, encoding='utf-8-sig')
impact_65.to_csv(os.path.join(base_dir, 'variable_impact_rank_2100_65countries.csv'), encoding='utf-8-sig')
print("✓ 65ヶ国版 CSV 保存完了:")
print("  thresholds_5regions_65countries.csv")
print("  thresholds_5regions_by_item_65countries.csv")
print("  thresholds_by_variable_5regions_65countries.csv")
print("  comparison_2100_vs_thresholds_65countries.csv")
print("  variable_impact_rank_2100_65countries.csv")

## 各地域ごとの閾値

地域 = **国（Area）× 穀物（Item）** の組み合わせとし、その地域の過去収量（Yield）から以下の閾値を算出します。

- **平均収量** (mean)：その地域の平年値の目安  
- **中央値** (median)：外れ値に強い基準値  
- **25%点** (p25)：低収量の目安（この以下は「低い」とみなす閾値）  
- **75%点** (p75)：高収量の目安（この以上は「高い」とみなす閾値）

In [ ]:
# 地域（Area × Item）ごとの収量閾値を算出
g = regressionDF.groupby(['Area', 'Item'])['Yield']
thresholds = pd.DataFrame({
    '閾値_平均': g.mean(),
    '閾値_中央値': g.median(),
    '閾値_25%点': g.quantile(0.25),
    '閾値_75%点': g.quantile(0.75),
    '観測年数': g.count()
}).reset_index()
thresholds = thresholds.round(2)
thresholds

,Area,Item,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測年数
0,Afghanistan,Maize (corn),1690.65,1650.00,1500.0,1696.30,53
1,Afghanistan,Rice,2237.15,2000.00,1904.8,2294.10,53
2,Afghanistan,Wheat,1228.08,1198.00,1025.0,1265.90,53
3,Albania,Maize (corn),3388.56,3341.60,2198.1,4226.30,53
4,Albania,Rice,3110.78,3157.30,2791.2,3500.00,33
...,...,...,...,...,...,...,...
353,Zambia,Rice,1027.94,1009.35,752.3,1278.85,46
354,Zambia,Wheat,4240.29,4246.50,2951.7,6118.10,53
355,Zimbabwe,Maize (corn),1247.64,1202.20,875.1,1707.90,53
356,Zimbabwe,Rice,1340.51,1300.00,643.8,1817.30,53


In [ ]:
# 国（Area）のみでまとめた閾値（穀物をまたいだ地域の目安）
g_area = regressionDF.groupby('Area')['Yield']
area_thresholds = pd.DataFrame({
    '閾値_平均': g_area.mean(),
    '閾値_中央値': g_area.median(),
    '閾値_25%点': g_area.quantile(0.25),
    '閾値_75%点': g_area.quantile(0.75),
    '観測数': g_area.count()
}).reset_index().round(2)
area_thresholds

,Area,閾値_平均,閾値_中央値,閾値_25%点,閾値_75%点,観測数
0,Afghanistan,1718.63,1659.80,1252.15,2000.00,159
1,Albania,3011.05,3019.70,2346.60,3683.35,139
2,Algeria,1706.40,1350.60,868.75,2435.30,159
3,Angola,908.34,870.20,666.70,1111.10,156
4,Antigua and Barbuda,1834.22,1705.90,1611.00,1960.00,47
...,...,...,...,...,...,...
151,Uruguay,2994.38,2565.30,1177.15,4243.40,159
152,Uzbekistan,3995.15,3630.50,2825.28,4488.42,66
153,Vanuatu,514.30,507.70,500.00,526.25,51
154,Zambia,2346.91,1674.35,1003.68,3063.22,152


In [ ]:
# 閾値テーブルをCSVで保存（地域＝国×穀物）
out_path = os.path.join(base_dir, 'region_yield_thresholds.csv')
thresholds.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"✓ 地域ごと閾値を保存しました: {out_path}")

# 国のみの閾値も保存
area_out_path = os.path.join(base_dir, 'area_yield_thresholds.csv')
area_thresholds.to_csv(area_out_path, index=False, encoding='utf-8-sig')
print(f"✓ 国ごと閾値を保存しました: {area_out_path}")

✓ 地域ごと閾値を保存しました: C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\region_yield_thresholds.csv
✓ 国ごと閾値を保存しました: C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\area_yield_thresholds.csv


In [7]:
# サマリー
print("=== 地域ごと閾値のサマリー ===")
print(f"地域数（国×穀物）: {len(thresholds)}")
print(f"国数: {thresholds['Area'].nunique()}")
print(f"穀物: {thresholds['Item'].unique().tolist()}")
print("\n閾値_中央値の分布:")
print(thresholds['閾値_中央値'].describe())

=== 地域ごと閾値のサマリー ===
地域数（国×穀物）: 358
国数: 156
穀物: ['Maize (corn)', 'Rice', 'Wheat']

閾値_中央値の分布:
count      358.000000
mean      2764.674441
std       2239.717127
min        266.800000
25%       1335.300000
50%       2037.125000
75%       3597.075000
max      20895.200000
Name: 閾値_中央値, dtype: float64
